In [ ]:
from helper import *
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import ast

In [ ]:
df = pd.read_csv("output/df_clean_with_llm_themes_strategies_v003.csv")

In [ ]:
df["llm_strategies_openai"] = df["llm_strategies_openai"].str.capitalize()

In [ ]:
import re

def parse_strategies(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []
    return re.findall(r'"([^"]+)"', x)

df["llm_strategies_openai"] = df["llm_strategies_openai"].apply(parse_strategies)

df["T"] = df["T"].replace("Persuasion", "Direct Persuasion")
df["T"] = df["T"].replace("Ambivalence", "Decisional Balance")


In [ ]:
# explode list -> one row per (user, category)

long = (
    df[["user_id_raw", "T", "llm_strategies_openai"]]
    .explode("llm_strategies_openai")
    .dropna(subset=["llm_strategies_openai"])
)

# count users per category by treatment
counts = (
    long.groupby(["llm_strategies_openai", "T"])["user_id_raw"]
    .nunique()
    .rename("count")
    .reset_index()
)

# counts already computed as before
# counts = long.groupby(["llm_strategies_openai", "T"])["user_id_raw"].nunique()...

# totals per treatment for shares
totals = df.groupby("T")["user_id_raw"].nunique()

# wide table for plotting (counts -> shares)
table = (
    counts.pivot(index="llm_strategies_openai", columns="T", values="count")
    .fillna(0)
    .div(totals, axis=1)
)


In [ ]:
table.index = table.index.str.capitalize()
table

In [ ]:


# order treatments and colors
t_order = ["Change Talk",  "Decisional Balance", "Direct Persuasion"]
table = table[[c for c in t_order if c in table.columns] + [c for c in table.columns if c not in t_order]]

table = table[table.index != "@analysis to=final code"]
    

# sort categories by Change Talk share (descending)
if "Change Talk" in table.columns:
    table = table.sort_values(by="Change Talk", ascending=True)

# plot
set_plot_theme()
fontsize = 18
plt.rcParams.update({
    "font.size": fontsize,
    "axes.titlesize": fontsize,
    "axes.labelsize": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize-2,
})
colors_by_t = {
    "Change Talk": plt.rcParams["axes.prop_cycle"].by_key()["color"][0],
    "Direct Persuasion": plt.rcParams["axes.prop_cycle"].by_key()["color"][1],
    "Decisional Balance": plt.rcParams["axes.prop_cycle"].by_key()["color"][2],
}
fig, ax = plt.subplots(figsize=(11, 8))

n_main = len(table)
n_split = len(table.columns)
group_height = 0.8
bar_height = group_height / n_split
y_positions = np.arange(n_main)

for i, t in enumerate(table.columns):
    values = table[t].values
    ypos = y_positions - group_height / 2 + i * bar_height + bar_height / 2
    ax.barh(
        ypos,
        values,
        height=bar_height,
        label=t,
        color=colors_by_t.get(t, None),
    )

ax.set_yticks(y_positions)
ax.set_yticklabels(table.index)
ax.set_xlabel("Share of participants mentioning strategy")
ax.legend(
    title=None,
    frameon=True,
    loc="lower right",
    fancybox=False,
    edgecolor="black",
    facecolor="white",
    framealpha=1.0,
)

sns.despine(ax=ax)

plt.savefig(
    "/path/to/project/code/analysis_NR/6731ca401220dcd3b28dc2ec/figures/fig_strategies.pdf",
    bbox_inches="tight",
)